## Import Libraries

In [1]:
import os
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split , KFold , GridSearchCV , RandomizedSearchCV , cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.metrics import classification_report , roc_auc_score , confusion_matrix , average_precision_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## Load dataset

In [2]:
df = pd.read_csv("fake_job_postings.csv")
df.head(3)

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0


## Explore Dataset

In [3]:
df.describe()

,job_id,telecommuting,has_company_logo,has_questions,fraudulent
count,17880.000000,17880.000000,17880.000000,17880.000000,17880.000000
mean,8940.500000,0.042897,0.795302,0.491723,0.048434
std,5161.655742,0.202631,0.403492,0.499945,0.214688
min,1.000000,0.000000,0.000000,0.000000,0.000000
25%,4470.750000,0.000000,1.000000,0.000000,0.000000
50%,8940.500000,0.000000,1.000000,0.000000,0.000000
75%,13410.250000,0.000000,1.000000,1.000000,0.000000
max,17880.000000,1.000000,1.000000,1.000000,1.000000


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17880 entries, 0 to 17879
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   job_id               17880 non-null  int64 
 1   title                17880 non-null  object
 2   location             17534 non-null  object
 3   department           6333 non-null   object
 4   salary_range         2868 non-null   object
 5   company_profile      14572 non-null  object
 6   description          17879 non-null  object
 7   requirements         15184 non-null  object
 8   benefits             10668 non-null  object
 9   telecommuting        17880 non-null  int64 
 10  has_company_logo     17880 non-null  int64 
 11  has_questions        17880 non-null  int64 
 12  employment_type      14409 non-null  object
 13  required_experience  10830 non-null  object
 14  required_education   9775 non-null   object
 15  industry             12977 non-null  object
 16  func

In [5]:
df.shape

(17880, 18)

In [6]:
df.isnull().sum()

job_id                     0
title                      0
location                 346
department             11547
salary_range           15012
company_profile         3308
description                1
requirements            2696
benefits                7212
telecommuting              0
has_company_logo           0
has_questions              0
employment_type         3471
required_experience     7050
required_education      8105
industry                4903
function                6455
fraudulent                 0
dtype: int64

In [7]:
# dropping unwanted or more missing values columns

col_to_drop = ["job_id","department","salary_range"]
df = df.drop(columns=(col_to_drop))

## Handle Missing Values

In [8]:
categorical_cols = ["location","employment_type","required_experience","required_education","industry","function"]
for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

text_cols = ["title","company_profile","description","requirements","benefits"]
for col in text_cols:
    df[col] = df[col].fillna(" ")
    df[f"{col}_length"] = df[col].apply(lambda x: len(str(x).split()))

In [9]:
df.isnull().sum()

title                     0
location                  0
company_profile           0
description               0
requirements              0
benefits                  0
telecommuting             0
has_company_logo          0
has_questions             0
employment_type           0
required_experience       0
required_education        0
industry                  0
function                  0
fraudulent                0
title_length              0
company_profile_length    0
description_length        0
requirements_length       0
benefits_length           0
dtype: int64

## Feature Selction

In [10]:
numeric_features = [
    "title_length","company_profile_length","description_length","requirements_length","benefits_length","telecommuting","has_company_logo","has_questions"
]

categorical_features = [
    "employment_type","required_experience","required_education","industry","function","location"
]

X = df[numeric_features + categorical_features]
y = df["fraudulent"]

## Preprocessor

In [11]:
numeric_transformer = Pipeline([
    ("Imputer",SimpleImputer(strategy='mean')),
    ("Scaling",StandardScaler())
])

categorical_tranformer = Pipeline([
    ("Imputer",SimpleImputer(strategy="most_frequent")),
    ("Onehot",OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("Num",numeric_transformer , numeric_features),
    ("Cat",categorical_tranformer , categorical_features)
])

In [12]:
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
kf = KFold(n_splits=5 , shuffle=True , random_state=42)

## Train Model

In [13]:
#Logistic Regression

log_reg = Pipeline([
    ("preprocessor",preprocessor),
    ("clf",LogisticRegression(max_iter=1000))
])

log_params = {"clf__C":[0.1,1,10]}
log_search = GridSearchCV(log_reg,param_grid=log_params,cv=kf,scoring="f1",n_jobs=-1)
log_search.fit(X_train,y_train)
print("Logistic Regression Beat Params ",log_search.best_params_)
print("ROC-AUC ",roc_auc_score(y_test,log_search.predict_proba(X_test)[:,1]))

Logistic Regression Beat Params  {'clf__C': 10}
ROC-AUC  0.9458247483094651


In [14]:
# Decision Tree 

dt = Pipeline([
    ("preprocessor",preprocessor),
    ("clf",DecisionTreeClassifier())
])

dt_params = {"clf__max_depth":[5,10,20,None]}
dt_search = GridSearchCV(dt,param_grid=dt_params,cv = kf,scoring="f1",n_jobs=-1)
dt_search.fit(X_train,y_train)
print("Decision Tree Best Params",dt_search.best_params_)
print(classification_report(y_test,dt_search.predict(X_test)))

Decision Tree Best Params {'clf__max_depth': None}
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      3403
           1       0.74      0.68      0.71       173

    accuracy                           0.97      3576
   macro avg       0.86      0.83      0.85      3576
weighted avg       0.97      0.97      0.97      3576



In [15]:
# Random Forest

rf = Pipeline([
    ("preprocessor",preprocessor),
    ("clf",RandomForestClassifier())
])

rf_params = {"clf__n_estimators":[50,100],"clf__max_depth":[10,None]}
rf_search = GridSearchCV(rf , param_grid=rf_params,cv=kf,scoring="f1",n_jobs=-1)
rf_search.fit(X_train,y_train)
print("RandomForest Best Params",rf_search.best_params_)
print(classification_report(y_test,rf_search.predict(X_test)))

RandomForest Best Params {'clf__max_depth': None, 'clf__n_estimators': 100}
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3403
           1       0.97      0.61      0.75       173

    accuracy                           0.98      3576
   macro avg       0.98      0.81      0.87      3576
weighted avg       0.98      0.98      0.98      3576



In [16]:
# Support Vector Machine

svm = Pipeline([
    ("preprocessor",preprocessor),
    ("clf",SVC(probability=True))
])

svm_params = {"clf__C":[0.1,1,10],"clf__kernel":["linear","rbf"]}
svm_search = GridSearchCV(svm , param_grid=svm_params,cv=kf,scoring="f1",n_jobs=-1)
svm_search.fit(X_train,y_train)
print("Support Vector Best Params",svm_search.best_params_)
print(classification_report(y_test,svm_search.predict(X_test)))
print("RODC-AUC",roc_auc_score(y_test,svm_search.predict_proba(X_test)[:,1]))

Support Vector Best Params {'clf__C': 10, 'clf__kernel': 'rbf'}
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      3403
           1       0.92      0.72      0.81       173

    accuracy                           0.98      3576
   macro avg       0.95      0.86      0.90      3576
weighted avg       0.98      0.98      0.98      3576

RODC-AUC 0.9642698808769549


In [17]:
# KNN

knn = Pipeline([
    ("preprocessor",preprocessor),
    ("clf",KNeighborsClassifier())
])

knn_params = {"clf__n_neighbors":[3,5,7]}
knn_search = GridSearchCV(knn,param_grid=knn_params,cv=kf,scoring="f1",n_jobs=-1)
knn_search.fit(X_train,y_train)
print("KNN Best Params : ",knn_search.best_params_)
print(classification_report(y_test,knn_search.predict(X_test)))

KNN Best Params :  {'clf__n_neighbors': 7}
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      3403
           1       0.84      0.58      0.68       173

    accuracy                           0.97      3576
   macro avg       0.91      0.79      0.84      3576
weighted avg       0.97      0.97      0.97      3576



In [18]:
# XGboost

xgb = Pipeline([
    ("preprocessor",preprocessor),
    ("clf",XGBClassifier(eval_metric="logloss"))
])

xgb_params = {"clf__n_estimator":[50,100],"clf__max_depth":[3,5]}
xgb_search = GridSearchCV(xgb,param_grid=xgb_params,cv=kf,scoring="f1",n_jobs=-1)
xgb_search.fit(X_train,y_train)
print("XGB best params",xgb_search.best_params_)
print(classification_report(y_test,xgb_search.predict(X_test)))
print("ROC-AUC",roc_auc_score(y_test,xgb_search.predict_proba(X_test)[:,1]))

XGB best params {'clf__max_depth': 5, 'clf__n_estimator': 50}
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3403
           1       0.94      0.64      0.76       173

    accuracy                           0.98      3576
   macro avg       0.96      0.82      0.87      3576
weighted avg       0.98      0.98      0.98      3576

ROC-AUC 0.9743680771301757


In [19]:
# Light GBM

lgbm = Pipeline([
    ("preprocessor",preprocessor),
    ("clf",LGBMClassifier())
])

lgbm_params = {"clf__n_estimator":[50,100],"clf__max_depth":[-1,10]}
lgbm_search = GridSearchCV(lgbm,param_grid=lgbm_params,cv=kf,scoring="f1",n_jobs=-1)
lgbm_search.fit(X_train,y_train)
print("LGBM best params",lgbm_search.best_params_)
print(classification_report(y_test,lgbm_search.predict(X_test)))
print("ROC-AUC",roc_auc_score(y_test,xgb_search.predict_proba(X_test)[:,1]))

[LightGBM] [Warning] Unknown parameter: n_estimator
[LightGBM] [Warning] Unknown parameter: n_estimator
[LightGBM] [Info] Number of positive: 693, number of negative: 13611
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001715 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1488
[LightGBM] [Info] Number of data points in the train set: 14304, number of used features: 241
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.048448 -> initscore=-2.977604
[LightGBM] [Info] Start training from score -2.977604
LGBM best params {'clf__max_depth': -1, 'clf__n_estimator': 50}
[LightGBM] [Warning] Unknown parameter: n_estimator
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3403
           1       0.96      0.65      0.78       173

    accuracy                           0.98      3576
   ma

# Checking Score

In [20]:
print("LogisticRegression Accuracy: ",log_search.score(X_test,y_test)*100,log_search.score(X_train,y_train)*100)
print("DEcision Tree Accuracy: ",dt_search.score(X_test,y_test)*100,log_search.score(X_train,y_train)*100)
print("RandomForest Accuracy: ",rf_search.score(X_test,y_test)*100,log_search.score(X_train,y_train)*100)
print("SVM Accuracy: ",svm_search.score(X_test,y_test)*100,log_search.score(X_train,y_train)*100)
print("KNN Accuracy: ",knn_search.score(X_test,y_test)*100,log_search.score(X_train,y_train)*100)
print("XGBoost Accuracy: ",xgb_search.score(X_test,y_test)*100,log_search.score(X_train,y_train)*100)
print("LightGBM Accuracy: ",lgbm_search.score(X_test,y_test)*100,log_search.score(X_train,y_train)*100)

LogisticRegression Accuracy:  56.71641791044776 79.55092221331195
DEcision Tree Accuracy:  70.69486404833837 79.55092221331195
RandomForest Accuracy:  75.177304964539 79.55092221331195
SVM Accuracy:  80.90614886731392 79.55092221331195
KNN Accuracy:  68.4931506849315 79.55092221331195
XGBoost Accuracy:  75.86206896551724 79.55092221331195
[LightGBM] [Warning] Unknown parameter: n_estimator
LightGBM Accuracy:  77.66323024054984 79.55092221331195


In [21]:
## So the LightGBM Model  accuracy is the best where training accuracy is 79% and testing accuracy is 77%

## Predictions

In [22]:
# Example new job posting (replace with real values)
new_data = pd.DataFrame([{
    "title_length": 20,
    "company_profile_length": 150,
    "description_length": 500,
    "requirements_length": 200,
    "benefits_length": 50,
    "telecommuting": 0,
    "has_company_logo": 1,
    "has_questions": 0,
    "employment_type": "Full-time",
    "required_experience": "Mid-Senior level",
    "required_education": "Bachelor's Degree",
    "industry": "Information Technology",
    "function": "Engineering",
    "location": "New York, NY"
}])

# Predict fraud or not
best_lgbm = lgbm_search.best_estimator_
pred = best_lgbm.predict(new_data)
proba = best_lgbm.predict_proba(new_data)[:,1]

print("Prediction:", pred[0])            # 0 = Real, 1 = Fraud
print("Fraud Probability:", proba[0])


[LightGBM] [Warning] Unknown parameter: n_estimator
[LightGBM] [Warning] Unknown parameter: n_estimator
Prediction: 0
Fraud Probability: 0.0016247878168507964


In [23]:
new_data1 = pd.DataFrame([{
    "title_length": 5,                      
    "company_profile_length": 0,            
    "description_length": 50,               
    "requirements_length": 10,              
    "benefits_length": 0,                   
    "telecommuting": 1,                    
    "has_company_logo": 0,                  
    "has_questions": 0,                     
    "employment_type": "Other",
    "required_experience": "Internship",    
    "required_education": "High School",    
    "industry": "Unknown",
    "function": "Other",
    "location": "Unknown"
}])

proba = best_lgbm.predict_proba(new_data1)[:,1]

print("Prediction:", pred[0])            # 0 = Real, 1 = Fraud
print("Fraud Probability:", proba[0])


[LightGBM] [Warning] Unknown parameter: n_estimator
Prediction: 0
Fraud Probability: 0.2763002138965084


# Save Model

In [24]:
joblib.dump(lgbm_search.best_estimator_,"best_model.pkl")
print("Lighgbm model ssaved as best model")

Lighgbm model ssaved as best model


In [25]:
df

,title,location,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,title_length,company_profile_length,description_length,requirements_length,benefits_length
0,Marketing Intern,"US, NY, New York","We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,,0,1,0,Other,Internship,Unknown,Unknown,Marketing,0,2,141,124,115,0
1,Customer Service - Cloud Video Production,"NZ, , Auckland","90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,Unknown,Marketing and Advertising,Customer Service,0,6,153,315,200,227
2,Commissioning Machinery Assistant (CMA),"US, IA, Wever",Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,,0,1,0,Unknown,Unknown,Unknown,Unknown,Unknown,0,4,141,50,164,0
3,Account Executive - Washington DC,"US, DC, Washington",Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0,5,85,346,176,97
4,Bill Review Manager,"US, FL, Fort Worth",SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0,3,207,168,89,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17875,Account Director - Distribution,"CA, ON, Toronto",Vend is looking for some awesome new talent to...,Just in case this is the first time you’ve vis...,To ace this role you:Will eat comprehensive St...,What can you expect from us?We have an open cu...,0,1,1,Full-time,Mid-Senior level,Unknown,Computer Software,Sales,0,4,290,226,180,161
17876,Payroll Accountant,"US, PA, Philadelphia",WebLinc is the e-commerce platform and service...,The Payroll Accountant will focus primarily on...,- B.A. or B.S. in Accounting- Desire to have f...,Health &amp; WellnessMedical planPrescription ...,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Internet,Accounting/Auditing,0,2,330,161,111,54
17877,Project Cost Control Staff Engineer - Cost Con...,"US, TX, Houston",We Provide Full Time Permanent Positions for m...,Experienced Project Cost Control Staff Enginee...,At least 12 years professional experience.Abil...,,0,0,0,Full-time,Unknown,Unknown,Unknown,Unknown,0,11,32,171,159,0
17878,Graphic Designer,"NG, LA, Lagos",,Nemsia Studios is looking for an experienced v...,1. Must be fluent in the latest versions of Co...,Competitive salary (compensation will be based...,0,0,1,Contract,Not Applicable,Professional,Graphic Design,Design,0,2,0,77,86,35
